BrM Nanostring Cosmx 6k (36 Samples, 3M cells)
=======================

In [ ]:
# ==============================================================================
# CosMx Brain Metastasis Spatial Transcriptomics
# ==============================================================================
# %pwd

# 加载相关工具包
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", None)

import numpy as np
import os
import matplotlib.pyplot as plt
import palantir
import warnings
import gc
import pickle
import random
import torch

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity


import anndata as ad

from scipy.stats import gaussian_kde, entropy
from scipy.sparse import csr_matrix
import seaborn as sns

import sklearn
print(sklearn.__version__)  # 应输出 1.3.2

import scanpy as sc
import pyucell as uc

# pip install squidpy==1.3.1 scanpy==1.9.3 anndata==0.9.0 numpy==1.23.5 scipy==1.9.3 pandas==1.5.3 scikit-image==0.19.3 tifffile==2022.10.10 pillow==9.4.0 matplotlib==3.6.3 seaborn==0.12.2 numba==0.56.4 networkx==2.8.8 scikit-learn==1.2.2 statsmodels==0.13.5 omnipath==1.0.7   # 旧版 squidpy 才可以读入 
import squidpy as sq
print(sq.__version__)

import celltypist

# from napari_spatialdata import Interactive
import spatialdata as sd
import spatialdata_io

from scipy.spatial import distance_matrix
from collections import defaultdict

from statannotations.Annotator import Annotator
from itertools import combinations  # 自动生成所有两两配对

warnings.filterwarnings('ignore')
sc.logging.print_header()


1.9.0
1.8.1


/home/dingrong/miniconda3/envs/R453/lib/python3.14/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


Package,Version
pandas,2.3.3
numpy,2.4.3
matplotlib,3.10.9
palantir,1.4.4
torch,2.11.0 (2.10.0)
networkx,3.6.1
scikit-learn,1.9.0
anndata,0.12.16
scipy,1.17.1
seaborn,0.13.2


In [2]:

# 固定所有随机种子
seed = 1234
random.seed(seed)
np.random.seed(seed)
sc.settings.seed = seed


In [3]:
ad.settings.allow_write_nullable_strings = True  # 开启允许保存字符串类型

fig_dir = "/home/dingrong/Project/Brain_Metastasis_Cosmx/BM_Cosmx_Py/Result"
# fig_dir = Path().resolve() / "BM_Cosmx_Py"/"Result"
os.makedirs(fig_dir, exist_ok=True)

# ===================== PLOT SETTINGS =====================
sc.set_figure_params(
	facecolor="white", figsize=(6,6), dpi=300, fontsize=11, frameon=True
)
sc.settings.verbosity = 3
sc.settings.n_jobs = 16
sc.settings.figdir = fig_dir

plt.rcParams["savefig.dpi"] = 300
# plt.rcParams["save.bbox_inches"] = "tight"
plt.rcParams['axes.unicode_minus'] = False


# ===================== UTILITY FUNCTIONS =====================
def save_fig(name, fig_dir):
	plt.savefig(os.path.join(fig_dir, f"{name}.png"), dpi=300, bbox_inches="tight" )
	plt.savefig(os.path.join(fig_dir, f"{name}.pdf"), dpi=300, bbox_inches="tight" )
	# plt.close()

# 自动缩放空间坐标
def auto_scale_spatial(adata, target_scale=1000):
	"""
	自动将空间坐标缩放到 [0, target_scale] 范围
	"""
	x = adata.obsm["spatial"][:, 0]
	y = adata.obsm["spatial"][:, 1]
	# 缩放到 0~1
	x = (x - x.min()) / (x.max() - x.min())
	y = (y - y.min()) / (y.max() - y.min())
	# 放大到目标范围
	x = x * target_scale
	y = y * target_scale
	# 放回
	adata.obsm["spatial"] = np.column_stack([x, y])

def plot_umap_spatial(adata, color_key, fig_dir, suffix=""):
	sc.pl.umap(adata, color=color_key, show=False)   # , save=f"umap_{color_key}{suffix}"
	save_fig(f"umap.{color_key}{suffix}", fig_dir)
	for sid in sample_list:
		sub = adata[adata.obs["sample"] == sid].copy()
		if sub.n_obs < 50:
			continue
		# 修复空间坐标
		# sub.obsm["spatial"] = sub.obs[["CenterX_global_px", "CenterY_global_px"]].values  # 直接提取坐标画出来图是反的，CosMx 仪器：Y 轴向下增大（图像坐标系，左上角为原点）， Matplotlib/squidpy：Y 轴向上增大（数学坐标系，左下角为原点）
		# ✅ 永久翻转 Y 坐标
		x = sub.obs["CenterX_global_px"].values
		y = sub.obs["CenterY_global_px"].values
		y_flipped = y.max() + y.min() - y
		sub.obsm["spatial"] = np.column_stack([x, y_flipped])
		# 即使有 obsm["spatial"]，squidpy 依然会强制检查 adata.uns["spatial"], 强制添加 squidpy 必须要的空结构，解决 KeyError
		sub.uns["spatial"] = {"images": {}}
		# # 自动缩放你的 sub 数据坐标
		# auto_scale_spatial(sub, target_scale=1000)
		# sq.pl.spatial_scatter(sub, color=color_key, size=1, img=False, frameon=False, shape=None)  # , library_key="fov" , palette="tab20" , save=f"spatial.{color_key}.{sid}.{suffix}" # img=False, NanoString CosMx 数据，没有组织病理学图像（H&E）
		sq.pl.spatial_scatter(sub, color=color_key, size=1, img=False, frameon=False, legend_fontsize=4, shape=None)  # , library_key="fov" , save=f"spatial.{color_key}.{sid}.{suffix}"  , squidpy 中 legend_loc 只能填：right / left / top / bottom / center ,  scanpy 支持 legend_loc='outside right'
		# ax.invert_yaxis()  # ✅ 临时翻转 Y 轴，匹配 CosMx 原始图像
		save_fig(f"spatial_{color_key}.{sid}{suffix}", fig_dir)

def plot_enrich_bar(df, x_col, y_col, title, name, fig_dir, topn=15):
	plt.figure(figsize=(10,6))
	df_top = df.head(topn).sort_values(x_col, ascending=True)
	sns.barplot(data=df_top, x=x_col, y=y_col)
	plt.title(title, fontsize=12)
	plt.tight_layout()
	save_fig(name, fig_dir)

def plot_boxplot(df, cols, name, fig_dir, title=""):
	plt.figure(figsize=(10,6))
	ax = df[cols].boxplot(grid=False)
	ax.grid(False)  # 去掉网格线
	plt.title(title)
	plt.tight_layout()
	save_fig(name, fig_dir)

样本信息
---------------------------------------------

In [ ]:
# ===================== GLOBAL CONFIG =====================
project_name = "BM_Cosmx_Py"
out_dir = "/home/dingrong/Project/Brain_Metastasis_Cosmx/BM_Cosmx_Py/Result/"
# out_dir = "/home/dingrong/Project/Brain_Metastasis_Cosmx/BM_Cosmx_Py/Result.bk/"
fig_dir = os.path.join(out_dir, "figures")
csv_dir = os.path.join(out_dir, "tables")
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

In [ ]:

adata = sc.read_h5ad(os.path.join(out_dir, f"{project_name}.h5ad"))
adata


【Step 9】 Cell Type Annotation from Seurat
---------------------------------------------

In [ ]:

# ==============================================================================
# METHODS: 9. Cell Type Annotation from Seurat
# ==============================================================================
print("\n【Step 9】 Cell Type Annotation")
fig_dir = os.path.join(out_dir, "Step09_Annotation")
csv_dir = os.path.join(out_dir, "Step09_Annotation")
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)



【Step 9】 Cell Type Annotation


In [ ]:

# 固定所有随机种子
seed = 1234
random.seed(seed)
np.random.seed(seed)
sc.settings.seed = seed


In [ ]:
CellType_colors = {
	"Astrocyte_PTPRZ1": "#33A02C",
	"Astrocyte_SLC1A2": "#BE9C2E",
	"Astrocyte_TNC": "#FFFF00",
	"Bcell": "#F7D42A",
	"Doublet": "#A8A6A7",
	"Endo_Stroma": "#6E8E84",
	"Endothelial": "#FF7F0E",
	"Fibroblast_COL7A1": "#AA9486",
	"Fibroblast_LUM": "#A6761D",
	"Macrophage_C1Q": "#AD8CFF",
	"Macrophage_CD163": "#86486F",
	"Macrophage_SPP1": "#6B42C8",
	"MCC_c0": "#FA41CA",
	"MCC_c1_EMT": "#FF7F00",
	"MCC_c10_Cycle_S": "#015E05",
	"MCC_c11_IFN": "#FCF40E",
	"MCC_c14_Cilium": "#8B5B45",
	"MCC_c2_Immune": "#007ED3",
	"MCC_c3": "#E50000",
	"MCC_c4_Hypoxic": "#894FC6",
	"MCC_c5_Cycle_G2M": "#7FD2FF",
	"MCC_c6_Cycle_S": "#00C19B",
	"MCC_c7_Mito": "#FEC3BB",
	"MCC_c9_Stress": "#49F149",
	"Microglial": "#68D359",
	"Monocyte": "#377EB8",
	"Mural": "#FFA88B",
	"Myeloid_Cycle": "#FF6AD5",
	"Neuron_Excitatory_LAMP5": "#FF00FF",
	"Neuron_Excitatory_RELN": "#FB832D",
	"Neuron_Excitatory_SYT1": "#C72E29",
	"Neuron_Inhibitory_CALB1": "#016392",
	"Neuron_Inhibitory_PVALB": "#8085E9",
	"Oligodendrocyte_CNP": "#0080FF",
	"Oligodendrocyte_MBP": "#EEAAEE",
	"Oligodendrocyte_NGFR": "#1ABC9C",
	"Oligodendrocyte_PLP1": "#47ECDD",
	"Oligodendrocyte_PRUNE2": "#9B59B6",
	"Tcell": "#30ACEC",
	"Undefined": "#B0B0B0"
}
categories = adata.obs["CellType"].cat.categories  # 拿到所有细胞类型顺序
color_list = [CellType_colors[ct] for ct in categories]  # 按顺序生成颜色
adata.uns["CellType_colors"] = color_list  

fig, ax = plt.subplots(figsize=(7, 3))
sc.pl.umap(adata, color="CellType", show=False, s=1, alpha=1, frameon=True, ax=ax)  # , legend_loc="on data" , save="umap_celltype"
ax.set_aspect("equal") # 强制xy等比例，UMAP坐标轴1:1
save_fig("umap_CellType_anno.adata", fig_dir)

# fig, ax = plt.subplots(figsize=(7, 3))
# sc.pl.umap(adata, color="sample", show=False, s=1, alpha=1, frameon=True, ax=ax)  # , legend_loc="on data" , save="umap_celltype"
# ax.set_aspect("equal") # 强制xy等比例，UMAP坐标轴1:1
# save_fig("umap_sample_anno.adata", fig_dir)


In [ ]:
# adata.obs["CellType_main"] = adata.obs["CellType_main"].astype("category")
CellType_main_colors = {
	"MCC": "#E31A1C",
	"Neuron": "#18BC9C",
	"Glial": "#CCDE8B",
	"Bcell": "#F7D42A",
	"Tcell": "#30ACEC",
	"Myeloid": "#6B42C8",
	"Stroma": "#A6761D"
}
categories = adata.obs["CellType_main"].cat.categories  # 拿到所有细胞类型顺序
color_list = [CellType_main_colors[ct] for ct in categories]  # 按顺序生成颜色
adata.uns["CellType_main_colors"] = color_list  

fig, ax = plt.subplots(figsize=(4, 3))
sc.pl.umap(adata, color="CellType_main", show=False, s=1, alpha=1, frameon=True, ax=ax)  # , legend_loc="on data" , save="umap_celltype"
ax.set_aspect("equal") # 强制xy等比例，UMAP坐标轴1:1
save_fig("umap_CellType_main_anno.adata", fig_dir)


【Step 9.2】 Cell Markers
---------------------------------------------

In [ ]:
# ==============================================================================
# METHODS: 9.2 Cell Markers check
# ==============================================================================
print("\n【Step 9.2】 Cell Markers")
fig_dir = os.path.join(out_dir, "Step09_2_Markers")
csv_dir = os.path.join(out_dir, "Step09_2_Markers")
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)



【Step 9.2】 Cell Markers


In [ ]:

# 固定所有随机种子
seed = 1234
random.seed(seed)
np.random.seed(seed)
sc.settings.seed = seed


In [ ]:
marker_dict = {
	"Markers": ["EPCAM", "KRT7", "KRT8", "KRT19", "CXCL17", "CCNL2", "VEGFA", "JUN", "IFI44L", "TPPP3", "AGR3", "ENO2", "SNAP25", "SYT1", "RELN", "LAMP5", "GAD1", "PVALB", "CALB1", "GFAP", "CLU", "PTPRZ1", "SLC1A2", "TNC", "S100B", "MOG", "PLP1", "MBP", "CNP", "CLDN11", "NGFR", "PRUNE2", "CD79A", "IGKC", "IGHA1", "CD3D", "TRAC", "GZMA", "CD163", "CD68", "C1QA", "SPP1", "CD83", "LYZ", "COL1A1", "LUM", "COL7A1", "ACTA2", "PECAM1", "FLT1", "CLDN5", "MKI67", "CDK1", "TOP2A"]
}

for cell_name, marker_list in marker_dict.items():
	print(f"正在绘制：{cell_name}")
	valid_markers  = [g for g in marker_list if g in adata.var_names]
	print(valid_markers )

	if len(valid_markers)==0:
		print(f"{cell_name} 无有效基因，跳过")
		continue
	
	sc.pl.dotplot(adata, var_names=valid_markers , groupby="CellType", standard_scale="var", show=False)  # , save="marker_dotplot"
	save_fig(f"CellType_marker_dotplot.{cell_name}", fig_dir)

	sc.pl.stacked_violin(adata, valid_markers, groupby="CellType", swap_axes=False, dendrogram=True, show=False) # valid_markers [:6]
	# sc.pl.violin(adata, valid_markers , groupby="CellType", show=False) # valid_markers [:6]
	save_fig(f"CellType_marker_violin.{cell_name}", fig_dir)
	plt.close()


正在绘制：Markers
['EPCAM', 'KRT7', 'KRT8', 'KRT19', 'CXCL17', 'VEGFA', 'JUN', 'IFI44L', 'ENO2', 'SNAP25', 'SYT1', 'RELN', 'GAD1', 'PVALB', 'CALB1', 'GFAP', 'CLU', 'PTPRZ1', 'SLC1A2', 'TNC', 'S100B', 'MOG', 'PLP1', 'MBP', 'CNP', 'NGFR', 'CD79A', 'IGKC', 'IGHA1', 'CD3D', 'TRAC', 'GZMA', 'CD163', 'CD68', 'C1QA', 'SPP1', 'CD83', 'LYZ', 'COL1A1', 'LUM', 'COL7A1', 'ACTA2', 'PECAM1', 'FLT1', 'CLDN5', 'MKI67', 'CDK1', 'TOP2A']
